# Men's T20 World Cup scorecards, 2022

Rebuild the original ESPNcricinfo exports, verify what the scorecards cover, then compare tournament batting and bowling leaders. Run the cells from this notebook’s directory.

In [1]:
from scorecards import build_tables, rankings

tables = build_tables()
for name, table in tables.items():
    print(f'{name}: {len(table):,} rows')

dim_match_summary: 45 rows
dim_players: 213 rows
fact_batting_summary: 699 rows
fact_bowling_summary: 500 rows


## Match results and scorecard coverage

The scorecard ID links batting and bowling performances to a fixture. Both team orderings in the raw innings point to the same game.

In [2]:
matches = tables['dim_match_summary']
batting = tables['fact_batting_summary']
bowling = tables['fact_bowling_summary']
print(f"{batting['match_id'].nunique()} of {len(matches)} fixtures have batting records")
matches.loc[~matches['match_id'].isin(batting['match_id']), ['matchDate', 'team1', 'team2', 'winner']]

42 of 45 fixtures have batting records


,matchDate,team1,team2,winner
20,"Oct 26, 2022",Afghanistan,New Zealand,abandoned
24,"Oct 28, 2022",Afghanistan,Ireland,abandoned
25,"Oct 28, 2022",Australia,England,abandoned


## Batting and bowling

A captain or wicketkeeper may appear with different role marks in different games. Inspect one player after removing those marks. The bowling table also provides legal balls so that partial overs add correctly.

In [3]:
batting.loc[batting['batsmanName'].eq('Jos Buttler'), ['match', 'runs', 'balls', 'out/not_out']]

,match,runs,balls,out/not_out
229,Afghanistan Vs England,18,18,out
320,Ireland Vs England,0,2,out
485,England Vs New Zealand,73,47,out
603,Sri Lanka Vs England,28,23,out
680,India Vs England,80,49,not_out
692,Pakistan Vs England,26,17,out


In [4]:
bowling.loc[bowling['overs'].astype(str).str.contains(r'\.'), ['match', 'bowlerName', 'overs', 'legal_balls']].head()

,match,bowlerName,overs,legal_balls
22,U.A.E. Vs Netherlands,Zawar Farid,2.5,17
33,Scotland Vs West Indies,Safyaan Sharif,3.3,21
55,Namibia Vs Netherlands,David Wiese,2.3,15
69,Sri Lanka Vs U.A.E.,Maheesh Theekshana,3.1,19
70,Sri Lanka Vs U.A.E.,Dushmantha Chameera,3.5,23


## Player profiles

Team and cleaned player name identify each profile. The checked-in player enrichment file supplies optional image URLs and alternate biographical text.

In [5]:
players = tables['dim_players']
players.loc[players['team'].eq('India'), ['name', 'team', 'battingStyle', 'playingRole']].head()

,name,team,battingStyle,playingRole
124,KL Rahul,India,Right hand Bat,Opening Batter
125,Rohit Sharma,India,Right hand Bat,Top order Batter
126,Virat Kohli,India,Right hand Bat,Top order Batter
127,Suryakumar Yadav,India,Right hand Bat,Batter
128,Axar Patel,India,Left hand Bat,Bowling Allrounder


## Tournament leaders

Rank batting by total runs and bowling by total wickets. Strike rate and economy use tournament totals, so a short innings does not carry the same weight as a long one.

In [6]:
batting_leaders, bowling_leaders = rankings(tables)
batting_leaders.head()

,teamInnings,batsmanName,innings,runs,balls,strike_rate
63,India,Virat Kohli,6,296,217,136.41
91,Netherlands,Max O'Dowd,8,242,215,112.56
62,India,Suryakumar Yadav,6,239,126,189.68
46,England,Jos Buttler,6,225,156,144.23
153,Sri Lanka,Kusal Mendis,8,223,156,142.95


In [7]:
bowling_leaders.head()

,bowlingTeam,bowlerName,wickets,runs,legal_balls,economy
105,Sri Lanka,Wanindu Hasaranga de Silva,15,199,186,6.42
59,Netherlands,Bas de Leede,13,169,132,7.68
36,England,Sam Curran,13,148,136,6.53
122,Zimbabwe,Blessing Muzarabani,12,199,156,7.65
89,South Africa,Anrich Nortje,11,94,105,5.37
